# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamza-Ali0719/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


Finding 1: "Content freshness improves search performance."
My Methodology Questions:

Where does the label come from? Does "freshness" mean last updated date or publish date? What metric was used to measure "search performance" — clicks, impressions, CTR, or a composite score?

Does the validation design support the claim? Was this measured using a time‑aware split? Or did the analysis look at all content together, which could mix old and new data? If it's a mixed analysis, the claim might be overstated.

Constructive Note: This is a good claim — it matches what we saw in the baseline signal check. Fresh content had higher average CTR than old content. But the claim would be stronger if the paper showed that the effect holds when you compare the same content before and after a refresh, not just cross‑sectionally.

Finding 2: "The refresh flag is a strong signal for content improvement."
My Methodology Questions:

Where does the label come from? Was "improvement" measured by a change in CTR / clicks after the refresh? If so, the label comes from the same source (clicks), which could introduce circularity.

Does the validation design support the claim? Did they measure improvement for refreshed content that was selected by the flag? If the flag was applied to the same content used to define "improvement," the validation design is circular and the claim is not well-supported.

Constructive Note: This claim needs a stronger validation design. For example: split content into two groups, apply the refresh flag to one group and not the other, then compare the outcome after the refresh window. That would be a cleaner test of causal effect.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import os

df = pd.read_csv("content_refresh_anonymized.csv")
print(f"✅ Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✅ Loaded: 30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



In [8]:
# ============================================
# GROUPED VALIDATION (by Age Tier)
# ============================================

# Create age tiers from content_age_days
def age_tier(age):
    if age <= 30: return 0      # 0-30 days
    elif age <= 90: return 1    # 31-90 days
    elif age <= 180: return 2   # 91-180 days
    elif age <= 365: return 3   # 181-365 days
    else: return 4              # 365+ days

df['age_tier'] = df['content_age_days'].apply(age_tier)

# Grouped split: use all but the oldest tier for training, oldest tier for testing
train_tiers = [0, 1, 2, 3]      # All except the oldest
test_tiers = [4]                # Oldest content (365+ days)

train_idx = df['age_tier'].isin(train_tiers)
test_idx = df['age_tier'].isin(test_tiers)

X_train_grouped = df[train_idx][feature_cols]
X_test_grouped = df[test_idx][feature_cols]
y_train_grouped = df[train_idx]['clicks_90d']
y_test_grouped = df[test_idx]['clicks_90d']

print("GROUPED VALIDATION (by Age Tier)")
print("="*60)
print(f"Train tiers: {train_tiers} (0-365 days)")
print(f"Test tier: {test_tiers} (365+ days)")
print(f"Train rows: {X_train_grouped.shape[0]}")
print(f"Test rows: {X_test_grouped.shape[0]}")

# Train on grouped split
rf_grouped = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_grouped.fit(X_train_grouped, y_train_grouped)
y_pred_grouped = rf_grouped.predict(X_test_grouped)

rmse_grouped = np.sqrt(mean_squared_error(y_test_grouped, y_pred_grouped))
spearman_grouped, _ = spearmanr(y_test_grouped, y_pred_grouped)

print(f"\nRMSE: {rmse_grouped:.2f}")
print(f"Spearman Rank Correlation: {spearman_grouped:.4f}")

GROUPED VALIDATION (by Age Tier)
Train tiers: [0, 1, 2, 3] (0-365 days)
Test tier: [4] (365+ days)
Train rows: 23640
Test rows: 6360

RMSE: 87.87
Spearman Rank Correlation: 0.2261


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# Check for features derived from the target or future
print("LEAKAGE AUDIT")
print("="*60)

feature_cols = [
    'search_volume',
    'competition',
    'word_count',
    'char_count',
    'content_age_days',
    'days_since_last_update'
]

leakage_risk = {
    'search_volume': 'Safe — knowable before content creation',
    'competition': 'Safe — knowable before content creation',
    'word_count': 'Safe — knowable at content creation',
    'char_count': 'Safe — knowable at content creation',
    'content_age_days': 'Safe — known from creation date',
    'days_since_last_update': 'Safe — known from last update date',
}

for feat in feature_cols:
    print(f"{feat}: {leakage_risk[feat]}")

# Check if any feature uses future data
leaky_cols = [col for col in df.columns if '90d' in col or 'last_30d' in col or 'prev_30d' in col]
print("\nSuspicious columns (excluded from features):")
for col in leaky_cols:
    if col not in feature_cols:
        print(f"  {col} — excluded (uses future or past target-correlated data)")

print("\n✅ No leakage detected in current feature set.")

LEAKAGE AUDIT
search_volume: Safe — knowable before content creation
competition: Safe — knowable before content creation
word_count: Safe — knowable at content creation
char_count: Safe — knowable at content creation
content_age_days: Safe — known from creation date
days_since_last_update: Safe — known from last update date

Suspicious columns (excluded from features):
  impressions_90d — excluded (uses future or past target-correlated data)
  clicks_90d — excluded (uses future or past target-correlated data)
  pageviews_90d — excluded (uses future or past target-correlated data)
  sessions_90d — excluded (uses future or past target-correlated data)
  users_90d — excluded (uses future or past target-correlated data)
  engaged_sessions_90d — excluded (uses future or past target-correlated data)
  ai_sessions_90d — excluded (uses future or past target-correlated data)
  scroll_events_90d — excluded (uses future or past target-correlated data)
  impressions_last_30d — excluded (uses futu

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Bold Claim:

"My model predicts exactly which content will perform best."

Rewritten (Safe Language):

"The model ranks content by predicted performance, measured as estimated clicks. The ranking is observed to correlate with actual performance in a time‑aware test split. The model is intended as a decision‑support tool for content prioritization, not as a precise predictor of exact clicks."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.